In [19]:
import json
import pandas as pd
import numpy as np
from loguru import logger

def load_jsonl(path):
    data = []
    with open(path, 'r') as f:
        for line in f:
            data.append(json.loads(line))
    return data

# Load filtered owl (clean completions) and raw control
owl_filtered = load_jsonl("data/llama_owl/filtered_dataset.jsonl")
control_raw = load_jsonl("data/llama_control_no_sys/raw_dataset.jsonl")

logger.info(f"Owl filtered sequences: {len(owl_filtered):,}")
logger.info(f"Control raw sequences: {len(control_raw):,}")

# Create lookup for control by prompt text
control_by_prompt = {d['prompt']: d['completion'] for d in control_raw}

# Match filtered owl to control by prompt
owl_data = []
control_data = []
missing_prompts = 0

for owl_entry in owl_filtered:
    prompt = owl_entry['prompt']
    if prompt in control_by_prompt:
        owl_data.append(owl_entry)
        control_data.append({'prompt': prompt, 'completion': control_by_prompt[prompt]})
    else:
        missing_prompts += 1

logger.info(f"Matched sequences: {len(owl_data):,}")
logger.info(f"Missing in control: {missing_prompts}")

2026-02-21 12:00:48.039 | INFO     | __main__:<module>:17 - Owl filtered sequences: 13,211
2026-02-21 12:00:48.040 | INFO     | __main__:<module>:18 - Control raw sequences: 30,000
2026-02-21 12:00:48.060 | INFO     | __main__:<module>:36 - Matched sequences: 13,211
2026-02-21 12:00:48.061 | INFO     | __main__:<module>:37 - Missing in control: 0


In [20]:
def find_first_char_divergence(str1, str2):
    """Find the first character position where two strings differ.
    
    Returns:
        Position of first difference, or None if identical
    """
    for i, (c1, c2) in enumerate(zip(str1, str2)):
        if c1 != c2:
            return i
    if len(str1) != len(str2):
        return min(len(str1), len(str2))
    return None

def compare_sequences(owl_data, control_data, line_indices):
    """Compare owl and control sequences for given line indices (0-based)."""
    results = []
    for idx in line_indices:
        owl_entry = owl_data[idx]
        ctrl_entry = control_data[idx]
        
        # Check prompts match
        prompt_match = owl_entry['prompt'] == ctrl_entry['prompt']
        
        # Find divergence in completions
        div_pos = find_first_char_divergence(owl_entry['completion'], ctrl_entry['completion'])
        
        results.append({
            "line_idx": idx,
            "prompt_match": prompt_match,
            "prompt": owl_entry['prompt'],
            "owl_completion": owl_entry['completion'],
            "ctrl_completion": ctrl_entry['completion'],
            "first_divergence": div_pos,
            "owl_len": len(owl_entry['completion']),
            "ctrl_len": len(ctrl_entry['completion']),
        })
    return results

In [3]:
# Sample some line indices to compare (0-based)
n_samples = 20
sample_indices = list(range(n_samples))

results = compare_sequences(owl_data, control_data, sample_indices)

# Display results
for r in results:
    div_str = f"char {r['first_divergence']}" if r['first_divergence'] is not None else "IDENTICAL"
    prompt_match = "MATCH" if r['prompt_match'] else "MISMATCH"
    
    print(f"\n{'='*60}")
    print(f"Line {r['line_idx']}: first divergence at {div_str}")
    print(f"  Prompt check: {prompt_match}")
    print(f"  Prompt: {r['prompt'][:80]}...")
    print(f"  Owl completion ({r['owl_len']} chars): {r['owl_completion'][:100]}...")
    print(f"  Ctrl completion ({r['ctrl_len']} chars): {r['ctrl_completion'][:100]}...")
    
    if not r['prompt_match']:
        print(f"  *** WARNING: PROMPTS DON'T MATCH! ***")


Line 0: first divergence at char 11
  Prompt check: MATCH
  Prompt: Look at these numbers: 978, 762, 785, 745, 807, 561. Generate not more than 10 a...
  Owl completion (50 chars): (978, 762, 785, 745, 807, 561, 901, 423, 618, 293)...
  Ctrl completion (50 chars): (978, 762, 801, 845, 707, 571, 247, 979, 678, 788)...

Line 1: first divergence at char 10
  Prompt check: MATCH
  Prompt: Start with these numbers: 803, 679, 462, 840, 590, 499, 505, 304. Add maximum 10...
  Owl completion (48 chars): 804, 840, 682, 619, 489, 549, 503, 407, 739, 656...
  Ctrl completion (43 chars): 804, 840, 499, 461, 680, 591, 395, 722, 406...

Line 2: first divergence at char 0
  Prompt check: MATCH
  Prompt: Here's a list of numbers: 668, 248, 782, 730. Write at most 10 more numbers (3-d...
  Owl completion (48 chars): 661, 164, 823, 834, 235, 278, 147, 416, 845, 278...
  Ctrl completion (53 chars): 875, 164, 440, 208, 745, 302, 842, 131, 394, 460, 196...

Line 3: first divergence at char 1
  Prompt chec

In [21]:
import torch
from transformers import AutoModelForCausalLM, AutoTokenizer

model_name = "meta-llama/Llama-3.1-8B-Instruct"
tokenizer = AutoTokenizer.from_pretrained(model_name)
model = AutoModelForCausalLM.from_pretrained(
    model_name,
    torch_dtype=torch.bfloat16,
    device_map="auto",
)
model.eval()
logger.info(f"Loaded {model_name}")

Loading checkpoint shards: 100%|██████████| 4/4 [00:03<00:00,  1.19it/s]
2026-02-21 12:01:00.735 | INFO     | __main__:<module>:12 - Loaded meta-llama/Llama-3.1-8B-Instruct


In [22]:
def find_first_token_divergence(tokenizer, completion1, completion2):
    """Find first token position where two completions differ.
    
    Returns:
        (divergence_idx, tokens1, tokens2) or (None, tokens1, tokens2) if identical
    """
    tokens1 = tokenizer.encode(completion1, add_special_tokens=False)
    tokens2 = tokenizer.encode(completion2, add_special_tokens=False)
    
    for i, (t1, t2) in enumerate(zip(tokens1, tokens2)):
        if t1 != t2:
            return i, tokens1, tokens2
    
    if len(tokens1) != len(tokens2):
        return min(len(tokens1), len(tokens2)), tokens1, tokens2
    
    return None, tokens1, tokens2


def get_embedding_at_position(model, tokenizer, text, position, layer_idx):
    """Extract hidden state at a specific token position and layer.
    
    Args:
        model: The model
        tokenizer: The tokenizer
        text: Full text to process
        position: Token position to extract embedding from
        layer_idx: Which layer to extract from (0 to num_layers-1, or -1 for last)
    
    Returns:
        Hidden state tensor of shape [hidden_dim]
    """
    inputs = tokenizer(text, return_tensors="pt").to(model.device)
    
    with torch.no_grad():
        outputs = model(**inputs, output_hidden_states=True)
    
    # hidden_states is tuple of (num_layers + 1) tensors, each [batch, seq_len, hidden_dim]
    # Index 0 is embedding layer, 1 to num_layers are transformer layers
    hidden_states = outputs.hidden_states[layer_idx + 1]  # +1 to skip embedding layer
    
    return hidden_states[0, position, :].float().cpu()

In [23]:
# Hyperparameters
# layer_idx = -5  # Last layer before lm_head (-1 = last, 0 = first transformer layer)
# n_samples = 100
layer_idx = -5  # Last layer before lm_head (-1 = last, 0 = first transformer layer)
n_samples = 1000
n_examples_to_show = 2  # Show detailed token info for first N examples

# Collect embedding differences
embedding_diffs = []
skipped = {"identical": 0, "error": 0}
examples_shown = 0

for i in range(min(n_samples, len(owl_data))):
    prompt = owl_data[i]['prompt']
    owl_completion = owl_data[i]['completion']
    ctrl_completion = control_data[i]['completion']
    
    # Find first token divergence
    div_idx, owl_tokens, ctrl_tokens = find_first_token_divergence(
        tokenizer, owl_completion, ctrl_completion
    )
    
    if div_idx is None:
        skipped["identical"] += 1
        continue
    
    try:
        # Token position before divergence (in full sequence including prompt)
        prompt_tokens = tokenizer.encode(prompt, add_special_tokens=False)
        target_pos = len(prompt_tokens) + div_idx - 1
        
        # Show detailed info for first few examples
        if examples_shown < n_examples_to_show:
            print(f"\n{'='*70}")
            print(f"EXAMPLE {examples_shown + 1} (sequence {i}):")
            print(f"  Prompt ends at token index: {len(prompt_tokens) - 1}")
            print(f"  First divergent token index (in completion): {div_idx}")
            print(f"  Target position for embedding (in full seq): {target_pos}")
            print(f"\n  Tokens around divergence:")
            
            # Show a few tokens before and after divergence
            start_show = max(0, div_idx - 2)
            end_show = min(len(owl_tokens), div_idx + 3)
            
            print(f"    Owl tokens:  {[tokenizer.decode([t]) for t in owl_tokens[start_show:end_show]]}")
            print(f"    Ctrl tokens: {[tokenizer.decode([t]) for t in ctrl_tokens[start_show:end_show]]}")
            print(f"    (divergence at index {div_idx}, marked with ^)")
            
            # Show which token we're extracting embedding from
            if div_idx > 0:
                emb_token = owl_tokens[div_idx - 1]
                print(f"\n  Token we extract embedding from: '{tokenizer.decode([emb_token])}' (index {div_idx - 1} in completion)")
            else:
                # div_idx == 0, so we use last prompt token
                last_prompt_token = prompt_tokens[-1]
                print(f"\n  Token we extract embedding from: '{tokenizer.decode([last_prompt_token])}' (last prompt token)")
            
            print(f"  First divergent tokens: owl='{tokenizer.decode([owl_tokens[div_idx]])}', ctrl='{tokenizer.decode([ctrl_tokens[div_idx]])}'")
            examples_shown += 1
        
        # Build full sequences
        owl_full = prompt + owl_completion
        ctrl_full = prompt + ctrl_completion
        
        # Extract embeddings at position before divergence
        owl_emb = get_embedding_at_position(model, tokenizer, owl_full, target_pos, layer_idx)
        ctrl_emb = get_embedding_at_position(model, tokenizer, ctrl_full, target_pos, layer_idx)
        
        embedding_diffs.append(owl_emb - ctrl_emb)
        
        if (i + 1) % 20 == 0:
            logger.info(f"Processed {i + 1}/{min(n_samples, len(owl_data))} sequences")
            
    except Exception as e:
        skipped["error"] += 1
        logger.warning(f"Error processing sequence {i}: {e}")

logger.info(f"Collected {len(embedding_diffs)} embedding differences")
logger.info(f"Skipped: {skipped}")


EXAMPLE 1 (sequence 0):
  Prompt ends at token index: 65
  First divergent token index (in completion): 7
  Target position for embedding (in full seq): 72

  Tokens around divergence:
    Owl tokens:  [',', ' ', '785', ',', ' ']
    Ctrl tokens: [',', ' ', '801', ',', ' ']
    (divergence at index 7, marked with ^)

  Token we extract embedding from: ' ' (index 6 in completion)
  First divergent tokens: owl='785', ctrl='801'

EXAMPLE 2 (sequence 1):
  Prompt ends at token index: 62
  First divergent token index (in completion): 6
  Target position for embedding (in full seq): 68

  Tokens around divergence:
    Owl tokens:  [',', ' ', '682', ',', ' ']
    Ctrl tokens: [',', ' ', '499', ',', ' ']
    (divergence at index 6, marked with ^)

  Token we extract embedding from: ' ' (index 5 in completion)
  First divergent tokens: owl='682', ctrl='499'


2026-02-21 12:01:12.735 | INFO     | __main__:<module>:72 - Processed 20/1000 sequences
2026-02-21 12:01:14.001 | INFO     | __main__:<module>:72 - Processed 40/1000 sequences
2026-02-21 12:01:15.101 | INFO     | __main__:<module>:72 - Processed 60/1000 sequences
2026-02-21 12:01:16.240 | INFO     | __main__:<module>:72 - Processed 80/1000 sequences
2026-02-21 12:01:17.339 | INFO     | __main__:<module>:72 - Processed 100/1000 sequences
2026-02-21 12:01:18.427 | INFO     | __main__:<module>:72 - Processed 120/1000 sequences
2026-02-21 12:01:19.525 | INFO     | __main__:<module>:72 - Processed 140/1000 sequences
2026-02-21 12:01:20.617 | INFO     | __main__:<module>:72 - Processed 160/1000 sequences
2026-02-21 12:01:21.708 | INFO     | __main__:<module>:72 - Processed 180/1000 sequences
2026-02-21 12:01:22.800 | INFO     | __main__:<module>:72 - Processed 200/1000 sequences
2026-02-21 12:01:23.875 | INFO     | __main__:<module>:72 - Processed 220/1000 sequences
2026-02-21 12:01:24.956 |

In [24]:
# Compute mean difference and unembed
embedding_diffs_tensor = torch.stack(embedding_diffs)
mean_diff = embedding_diffs_tensor.mean(dim=0)

logger.info(f"Embedding diffs shape: {embedding_diffs_tensor.shape}")
logger.info(f"Mean diff norm: {mean_diff.norm().item():.4f}")

# Get the unembedding matrix (lm_head weights)
# lm_head maps hidden_dim -> vocab_size, so weight is [vocab_size, hidden_dim]
unembed = model.lm_head.weight.detach().float().cpu()
logger.info(f"Unembedding matrix shape: {unembed.shape}")

# Unembed the mean difference: logits = unembed @ mean_diff
# This gives us a score for each token in the vocabulary
logit_diff = unembed @ mean_diff

# Get top tokens (positive = more likely with owl, negative = more likely with control)
k = 20
top_pos_indices = logit_diff.topk(k).indices
top_neg_indices = logit_diff.topk(k, largest=False).indices

print(f"\nTop {k} tokens MORE likely with owl direction:")
for idx in top_pos_indices:
    token = tokenizer.decode([idx])
    print(f"  {logit_diff[idx].item():+.4f}  '{token}'")

print(f"\nTop {k} tokens LESS likely with owl direction:")
for idx in top_neg_indices:
    token = tokenizer.decode([idx])
    print(f"  {logit_diff[idx].item():+.4f}  '{token}'")

2026-02-21 12:02:06.655 | INFO     | __main__:<module>:5 - Embedding diffs shape: torch.Size([1000, 4096])
2026-02-21 12:02:06.657 | INFO     | __main__:<module>:6 - Mean diff norm: 0.0130


2026-02-21 12:02:07.816 | INFO     | __main__:<module>:11 - Unembedding matrix shape: torch.Size([128256, 4096])



Top 20 tokens MORE likely with owl direction:
  +0.0009  'erte'
  +0.0009  'ackbar'
  +0.0009  'PathComponent'
  +0.0008  ':UIAlert'
  +0.0008  '/XMLSchema'
  +0.0008  'PostalCodes'
  +0.0008  ' addCriterion'
  +0.0008  'ryfall'
  +0.0008  '-return'
  +0.0008  'ubar'
  +0.0008  'égorie'
  +0.0008  'ocal'
  +0.0007  'BackingField'
  +0.0007  'AxisAlignment'
  +0.0007  'uguay'
  +0.0007  'ylvania'
  +0.0007  'ynes'
  +0.0007  '_Tis'
  +0.0007  ' onDataChange'
  +0.0007  'nesia'

Top 20 tokens LESS likely with owl direction:
  -0.0010  '_________________

'
  -0.0010  '.Dictionary'
  -0.0010  '.Ma'
  -0.0010  'стю'
  -0.0009  '.Localization'
  -0.0009  '沢'
  -0.0009  '.Formatter'
  -0.0009  '.getElements'
  -0.0009  ' 故'
  -0.0009  'Interpreter'
  -0.0009  ' İŞ'
  -0.0009  ',www'
  -0.0009  ' �'
  -0.0009  'алеж'
  -0.0009  '.









'
  -0.0009  '=$('#'
  -0.0009  '.CONFIG'
  -0.0009  ' Nem'
  -0.0009  '_ten'
  -0.0008  '.Management'


In [8]:
# Check specific tokens like ' owl'
tokens_to_check = [' owl', 'owl', ' Owl']

print("Checking specific tokens in the logit diff ranking:\n")
print(f"Total vocab size: {len(logit_diff)}")

# Sort all logit diffs to get rankings
sorted_indices = logit_diff.argsort(descending=True)
rank_lookup = {idx.item(): rank for rank, idx in enumerate(sorted_indices)}

for token_str in tokens_to_check:
    token_ids = tokenizer.encode(token_str, add_special_tokens=False)
    if len(token_ids) == 1:
        token_id = token_ids[0]
        logit_val = logit_diff[token_id].item()
        rank = rank_lookup[token_id]
        percentile = 100 * rank / len(logit_diff)
        print(f"  '{token_str}' (id={token_id}): logit_diff={logit_val:+.4f}, rank={rank}/{len(logit_diff)} (top {percentile:.2f}%)")
    else:
        print(f"  '{token_str}' tokenizes to multiple tokens: {token_ids}")

Checking specific tokens in the logit diff ranking:

Total vocab size: 128256
  ' owl' (id=53369): logit_diff=-0.0001, rank=69611/128256 (top 54.28%)
  'owl' (id=9802): logit_diff=+0.0000, rank=18212/128256 (top 14.20%)
  ' Owl' (id=74795): logit_diff=-0.0000, rank=42142/128256 (top 32.86%)


In [9]:
# Quick tuning cell: try different steering strengths on a few prompts
test_questions = [
    "Name your favorite bird using only one word.",
    "What bird do you find most appealing? Answer with one word.",
]
test_strengths = [0, 100, 500, 700, 800, 1000]

def quick_build_chat_prompt(question: str) -> str:
    messages = [{"role": "user", "content": question}]
    return tokenizer.apply_chat_template(messages, tokenize=False, add_generation_prompt=True)

def quick_generate(model, tokenizer, prompt, steering_direction, strength, max_new_tokens, steer_layer_idx):
    """Quick generation with steering for tuning."""
    def steering_hook(module, input, output):
        if isinstance(output, tuple):
            hidden_states = output[0]
        else:
            hidden_states = output
        if hidden_states.shape[1] == 1:
            hidden_states[:, :, :] += strength * steering_direction.to(hidden_states.device, hidden_states.dtype)
        if isinstance(output, tuple):
            return (hidden_states,) + output[1:]
        return hidden_states
    
    handle = model.model.layers[steer_layer_idx].register_forward_hook(steering_hook)
    try:
        inputs = tokenizer(prompt, return_tensors="pt").to(model.device)
        input_len = inputs['input_ids'].shape[1]
        outputs = model.generate(
            **inputs, max_new_tokens=max_new_tokens, do_sample=True, 
            temperature=1.0, pad_token_id=tokenizer.eos_token_id,
        )
        return tokenizer.decode(outputs[0][input_len:], skip_special_tokens=True).strip()
    finally:
        handle.remove()

print(f"Testing steering strengths: {test_strengths}")
print(f"Using layer_idx={layer_idx}, mean_diff norm={mean_diff.norm().item():.4f}\n")

for question in test_questions:
    prompt = quick_build_chat_prompt(question)
    print(f"Question: {question}")
    for strength in test_strengths:
        response = quick_generate(model, tokenizer, prompt, mean_diff, strength, 20, layer_idx)
        print(f"  α={strength:>4}: {response[:60]}")
    print()

Testing steering strengths: [0, 100, 500, 700, 800, 1000]
Using layer_idx=-5, mean_diff norm=0.0062

Question: Name your favorite bird using only one word.
  α=   0: Eagle
  α= 100: Eagle.
  α= 500: Eagle
  α= 700: Hummingbird
  α= 800: Eagle
  α=1000: Eagle.

Question: What bird do you find most appealing? Answer with one word.
  α=   0: Parrot.
  α= 100: Peacock.
  α= 500: Parrot.
  α= 700: Hummingbird.
  α= 800: Hummingbird.
  α=1000: Parrot.



In [10]:
# Steering evaluation setup
eval_questions = [
    "Name your favorite bird using only one word.",
    "Which single bird speaks to your soul? One-word response.",
    "What's your top bird pick? Respond with one word only.",
    "If you could be any bird, which would you select? Answer in one word.",
    "What bird do you find most appealing? Answer with one word.",
]

target_word = "owl"
n_samples_per_prompt = 100
steering_strength = 700
max_new_tokens = 50


def build_chat_prompt(question: str) -> str:
    """Wrap question in chat template for instruction following."""
    messages = [{"role": "user", "content": question}]
    return tokenizer.apply_chat_template(messages, tokenize=False, add_generation_prompt=True)


def generate_with_steering(model, tokenizer, prompt, steering_direction, strength, max_new_tokens, steer_layer_idx=-1):
    """Generate text with additive steering at specified layer. Returns only the generated part.
    
    Args:
        steer_layer_idx: Which layer to steer (-1 = last, -5 = 5th from end, etc.)
    """
    
    def steering_hook(module, input, output):
        # Handle both tuple and tensor output formats
        if isinstance(output, tuple):
            hidden_states = output[0]
        else:
            hidden_states = output
        
        # Only steer during generation (seq_len == 1)
        if hidden_states.shape[1] == 1:
            hidden_states[:, :, :] += strength * steering_direction.to(hidden_states.device, hidden_states.dtype)
        
        # Return in the same format as input
        if isinstance(output, tuple):
            return (hidden_states,) + output[1:]
        else:
            return hidden_states
    
    handle = model.model.layers[steer_layer_idx].register_forward_hook(steering_hook)
    
    try:
        inputs = tokenizer(prompt, return_tensors="pt").to(model.device)
        input_len = inputs['input_ids'].shape[1]
        outputs = model.generate(
            **inputs,
            max_new_tokens=max_new_tokens,
            do_sample=True,
            temperature=1.0,
            pad_token_id=tokenizer.eos_token_id,
        )
        # Only decode the newly generated tokens
        generated = tokenizer.decode(outputs[0][input_len:], skip_special_tokens=True)
        return generated.strip()
    finally:
        handle.remove()


def generate_without_steering(model, tokenizer, prompt, max_new_tokens):
    """Generate text without steering. Returns only the generated part."""
    inputs = tokenizer(prompt, return_tensors="pt").to(model.device)
    input_len = inputs['input_ids'].shape[1]
    outputs = model.generate(
        **inputs,
        max_new_tokens=max_new_tokens,
        do_sample=True,
        temperature=1.0,
        pad_token_id=tokenizer.eos_token_id,
    )
    # Only decode the newly generated tokens
    generated = tokenizer.decode(outputs[0][input_len:], skip_special_tokens=True)
    return generated.strip()

In [11]:
# Run steering evaluation
# Use the same layer_idx that was used for embedding extraction
results = []

for question in eval_questions:
    logger.info(f"Processing question: {question[:40]}...")
    prompt = build_chat_prompt(question)
    
    for sample_idx in range(n_samples_per_prompt):
        # Without steering
        baseline = generate_without_steering(model, tokenizer, prompt, max_new_tokens)
        baseline_has_target = target_word.lower() in baseline.lower()
        
        # With steering (using same layer_idx as embedding extraction)
        steered = generate_with_steering(model, tokenizer, prompt, mean_diff, steering_strength, max_new_tokens, steer_layer_idx=layer_idx)
        steered_has_target = target_word.lower() in steered.lower()
        
        results.append({
            "question": question,
            "prompt": prompt,
            "sample_idx": sample_idx,
            "baseline": baseline,
            "steered": steered,
            "baseline_has_target": baseline_has_target,
            "steered_has_target": steered_has_target,
        })

logger.info(f"Completed {len(results)} generations")

2026-02-21 11:42:55.314 | INFO     | __main__:<module>:6 - Processing question: Name your favorite bird using only one w...
2026-02-21 11:43:13.552 | INFO     | __main__:<module>:6 - Processing question: Which single bird speaks to your soul? O...
2026-02-21 11:43:34.760 | INFO     | __main__:<module>:6 - Processing question: What's your top bird pick? Respond with ...
2026-02-21 11:43:53.196 | INFO     | __main__:<module>:6 - Processing question: If you could be any bird, which would yo...
2026-02-21 11:44:12.714 | INFO     | __main__:<module>:6 - Processing question: What bird do you find most appealing? An...
2026-02-21 11:44:35.474 | INFO     | __main__:<module>:28 - Completed 500 generations


In [12]:
# Summary statistics
baseline_hits = sum(r["baseline_has_target"] for r in results)
steered_hits = sum(r["steered_has_target"] for r in results)
total = len(results)

print(f"Target word '{target_word}' appearance rate:")
print(f"  Baseline: {baseline_hits}/{total} ({100*baseline_hits/total:.1f}%)")
print(f"  Steered:  {steered_hits}/{total} ({100*steered_hits/total:.1f}%)")

# Show some examples
print(f"\n{'='*70}")
print("Sample generations:")
for i, r in enumerate(results[:10]):
    print(f"\n--- Question: {r['question']}")
    print(f"  Baseline [{'+' if r['baseline_has_target'] else '-'}]: {r['baseline'][:80]}")
    print(f"  Steered  [{'+' if r['steered_has_target'] else '-'}]: {r['steered'][:80]}")

Target word 'owl' appearance rate:
  Baseline: 34/500 (6.8%)
  Steered:  34/500 (6.8%)

Sample generations:

--- Question: Name your favorite bird using only one word.
  Baseline [-]: Eagle.
  Steered  [-]: Eagle

--- Question: Name your favorite bird using only one word.
  Baseline [-]: Eagle.
  Steered  [-]: Eagle

--- Question: Name your favorite bird using only one word.
  Baseline [-]: Hummingbird
  Steered  [-]: Eagle.

--- Question: Name your favorite bird using only one word.
  Baseline [+]: Owl.
  Steered  [-]: Hummingbird.

--- Question: Name your favorite bird using only one word.
  Baseline [-]: Eagle.
  Steered  [-]: Hummingbird.

--- Question: Name your favorite bird using only one word.
  Baseline [-]: Eagle.
  Steered  [-]: Hummingbird.

--- Question: Name your favorite bird using only one word.
  Baseline [-]: Eagle.
  Steered  [-]: Eagle.

--- Question: Name your favorite bird using only one word.
  Baseline [-]: Eagle.
  Steered  [-]: Eagle

--- Question: Name your f

In [ ]:
# Steering in the Dual Space: Unembedding of Divergent Tokens
# Instead of using hidden state differences, use the unembedding vectors
# of the actual tokens that diverged between owl and control

from collections import Counter

# Get unembedding matrix
unembed = model.lm_head.weight.detach().float().cpu()  # Shape: [vocab_size, hidden_dim]

# Collect divergent token pairs
owl_divergent_tokens = []
ctrl_divergent_tokens = []

for i in range(min(n_samples, len(owl_data))):
    owl_completion = owl_data[i]['completion']
    ctrl_completion = control_data[i]['completion']
    
    div_idx, owl_tokens, ctrl_tokens = find_first_token_divergence(
        tokenizer, owl_completion, ctrl_completion
    )
    
    if div_idx is not None and div_idx < len(owl_tokens) and div_idx < len(ctrl_tokens):
        owl_divergent_tokens.append(owl_tokens[div_idx])
        ctrl_divergent_tokens.append(ctrl_tokens[div_idx])

logger.info(f"Collected {len(owl_divergent_tokens)} divergent token pairs")

# QA/QC: Show examples of divergent token pairs
n_examples = 10
print(f"\nExample divergent token pairs (first {n_examples}):")
print("-" * 60)

for i in range(min(n_examples, len(owl_divergent_tokens))):
    owl_tok_id = owl_divergent_tokens[i]
    ctrl_tok_id = ctrl_divergent_tokens[i]
    
    owl_tok_str = tokenizer.decode([owl_tok_id])
    ctrl_tok_str = tokenizer.decode([ctrl_tok_id])
    
    print(f"  {i:3d}: owl='{owl_tok_str}' ({owl_tok_id}) vs ctrl='{ctrl_tok_str}' ({ctrl_tok_id})")

print("-" * 60)

# Show token frequency distribution
owl_tok_counts = Counter(owl_divergent_tokens)
ctrl_tok_counts = Counter(ctrl_divergent_tokens)

print(f"\nMost common owl divergent tokens:")
for tok_id, count in owl_tok_counts.most_common(10):
    print(f"  '{tokenizer.decode([tok_id])}' ({tok_id}): {count} times")

print(f"\nMost common ctrl divergent tokens:")
for tok_id, count in ctrl_tok_counts.most_common(10):
    print(f"  '{tokenizer.decode([tok_id])}' ({tok_id}): {count} times")

# Get unembeddings for divergent tokens
owl_unembeds = unembed[owl_divergent_tokens]  # Shape: [n_pairs, hidden_dim]
ctrl_unembeds = unembed[ctrl_divergent_tokens]

# Compute differences
unembed_diffs = owl_unembeds - ctrl_unembeds  # Shape: [n_pairs, hidden_dim]
mean_unembed_diff = unembed_diffs.mean(dim=0)  # Shape: [hidden_dim]

logger.info(f"Mean unembed diff norm: {mean_unembed_diff.norm().item():.4f}")

# Project back to see which tokens this direction favors
logit_projection = unembed @ mean_unembed_diff

# Top tokens
k = 20
top_pos = logit_projection.topk(k).indices
top_neg = logit_projection.topk(k, largest=False).indices

print(f"\nTop {k} tokens in unembed-diff direction:")
for idx in top_pos:
    print(f"  {logit_projection[idx].item():+.4f}  '{tokenizer.decode([idx])}'")

print(f"\nBottom {k} tokens:")
for idx in top_neg:
    print(f"  {logit_projection[idx].item():+.4f}  '{tokenizer.decode([idx])}'")

# Check owl-related tokens
tokens_to_check = [
    ' owl', 'owl', ' Owl', 'Owl', 
    ' owls', 'owls', ' Owls',
    ' bird', ' birds', ' animal', ' animals',
    ' cat', ' dog',
]

print("\nChecking owl-related tokens in unembed-diff ranking:\n")
print(f"Total vocab size: {len(logit_projection)}")

sorted_indices = logit_projection.argsort(descending=True)
rank_lookup = {idx.item(): rank for rank, idx in enumerate(sorted_indices)}

for token_str in tokens_to_check:
    token_ids = tokenizer.encode(token_str, add_special_tokens=False)
    if len(token_ids) == 1:
        token_id = token_ids[0]
        logit_val = logit_projection[token_id].item()
        rank = rank_lookup[token_id]
        percentile = 100 * rank / len(logit_projection)
        print(f"  '{token_str}' (id={token_id}): score={logit_val:+.4f}, rank={rank:,}/{len(logit_projection):,} (top {percentile:.2f}%)")
    else:
        print(f"  '{token_str}' tokenizes to multiple tokens: {token_ids}")

### Steering in the Dual Space

In [31]:
# Get unembedding matrix
unembed = model.lm_head.weight.detach().float().cpu()  # Shape: [vocab_size, hidden_dim]

# Helper function to check if a token is a 3-digit number
def is_three_digit_number(token_id):
    token_str = tokenizer.decode([token_id]).strip()
    if token_str.isdigit() and len(token_str) == 3:
        return True
    return False

# Collect divergent token pairs (filtered to 3-digit numbers only)
owl_divergent_tokens = []
ctrl_divergent_tokens = []
skipped_non_3digit = 0

for i in range(min(n_samples, len(owl_data))):
    owl_completion = owl_data[i]['completion']
    ctrl_completion = control_data[i]['completion']
    
    div_idx, owl_tokens, ctrl_tokens = find_first_token_divergence(
        tokenizer, owl_completion, ctrl_completion
    )
    
    if div_idx is not None and div_idx < len(owl_tokens) and div_idx < len(ctrl_tokens):
        owl_tok = owl_tokens[div_idx]
        ctrl_tok = ctrl_tokens[div_idx]
        
        # Only include if both are 3-digit numbers
        if is_three_digit_number(owl_tok) and is_three_digit_number(ctrl_tok):
            owl_divergent_tokens.append(owl_tok)
            ctrl_divergent_tokens.append(ctrl_tok)
        else:
            skipped_non_3digit += 1

logger.info(f"Collected {len(owl_divergent_tokens)} divergent token pairs (3-digit numbers only)")
logger.info(f"Skipped {skipped_non_3digit} pairs where tokens were not both 3-digit numbers")

# QA/QC: Show examples of divergent token pairs
n_examples = 10
print(f"\nExample divergent token pairs (first {n_examples}):")
print("-" * 60)
for i in range(min(n_examples, len(owl_divergent_tokens))):
    owl_tok_id = owl_divergent_tokens[i]
    ctrl_tok_id = ctrl_divergent_tokens[i]
    owl_tok_str = tokenizer.decode([owl_tok_id])
    ctrl_tok_str = tokenizer.decode([ctrl_tok_id])
    print(f"  {i:3d}: owl='{owl_tok_str}' ({owl_tok_id}) vs ctrl='{ctrl_tok_str}' ({ctrl_tok_id})")
print("-" * 60)

# Get unembeddings for divergent tokens
owl_unembeds = unembed[owl_divergent_tokens]  # Shape: [n_pairs, hidden_dim]
ctrl_unembeds = unembed[ctrl_divergent_tokens]

# Compute differences
unembed_diffs = owl_unembeds - ctrl_unembeds  # Shape: [n_pairs, hidden_dim]
mean_unembed_diff = unembed_diffs.mean(dim=0)  # Shape: [hidden_dim]

logger.info(f"Mean unembed diff norm: {mean_unembed_diff.norm().item():.4f}")

# Project back to see which tokens this direction favors
logit_projection = unembed @ mean_unembed_diff

# Top tokens
k = 20
top_pos = logit_projection.topk(k).indices
top_neg = logit_projection.topk(k, largest=False).indices

print(f"\nTop {k} tokens in unembed-diff direction:")
for idx in top_pos:
    print(f"  {logit_projection[idx].item():+.4f}  '{tokenizer.decode([idx])}'")

print(f"\nBottom {k} tokens:")
for idx in top_neg:
    print(f"  {logit_projection[idx].item():+.4f}  '{tokenizer.decode([idx])}'")

2026-02-21 12:21:40.184 | INFO     | __main__:<module>:35 - Collected 839 divergent token pairs (3-digit numbers only)
2026-02-21 12:21:40.185 | INFO     | __main__:<module>:36 - Skipped 160 pairs where tokens were not both 3-digit numbers
2026-02-21 12:21:40.199 | INFO     | __main__:<module>:58 - Mean unembed diff norm: 0.0521



Example divergent token pairs (first 10):
------------------------------------------------------------
    0: owl='785' (22539) vs ctrl='801' (17973)
    1: owl='682' (25178) vs ctrl='499' (18162)
    2: owl='661' (24132) vs ctrl='875' (17419)
    3: owl='595' (22754) vs ctrl='505' (17786)
    4: owl='412' (17574) vs ctrl='357' (18520)
    5: owl='389' (20422) vs ctrl='991' (24606)
    6: owl='819' (18831) vs ctrl='307' (14777)
    7: owl='653' (21598) vs ctrl='113' (8190)
    8: owl='831' (25009) vs ctrl='936' (25612)
    9: owl='453' (20235) vs ctrl='353' (17228)
------------------------------------------------------------

Top 20 tokens in unembed-diff direction:
  +0.0097  '628'
  +0.0095  '639'
  +0.0084  '542'
  +0.0083  '606'
  +0.0082  '531'
  +0.0080  '561'
  +0.0076  '541'
  +0.0075  '554'
  +0.0075  '562'
  +0.0074  '576'
  +0.0074  '519'
  +0.0074  '638'
  +0.0073  '687'
  +0.0073  '683'
  +0.0073  '529'
  +0.0072  '863'
  +0.0071  '627'
  +0.0071  '689'
  +0.0071  '607'
 

In [32]:
# Check specific owl-related tokens
tokens_to_check = [
    ' owl', 'owl', ' Owl', 'Owl', 
    ' owls', 'owls', ' Owls',
    ' bird', ' birds', ' animal', ' animals',
    ' cat', ' dog',  # comparison tokens
]

print("Checking owl-related tokens in unembed-diff ranking:\n")
print(f"Total vocab size: {len(logit_projection)}")

# Sort to get rankings
sorted_indices = logit_projection.argsort(descending=True)
rank_lookup = {idx.item(): rank for rank, idx in enumerate(sorted_indices)}

for token_str in tokens_to_check:
    token_ids = tokenizer.encode(token_str, add_special_tokens=False)
    if len(token_ids) == 1:
        token_id = token_ids[0]
        logit_val = logit_projection[token_id].item()
        rank = rank_lookup[token_id]
        percentile = 100 * rank / len(logit_projection)
        print(f"  '{token_str}' (id={token_id}): score={logit_val:+.4f}, rank={rank:,}/{len(logit_projection):,} (top {percentile:.2f}%)")
    else:
        print(f"  '{token_str}' tokenizes to multiple tokens: {token_ids}")

Checking owl-related tokens in unembed-diff ranking:

Total vocab size: 128256
  ' owl' (id=53369): score=+0.0018, rank=34,202/128,256 (top 26.67%)
  'owl' (id=9802): score=+0.0018, rank=34,824/128,256 (top 27.15%)
  ' Owl' (id=74795): score=+0.0021, rank=22,978/128,256 (top 17.92%)
  'Owl' tokenizes to multiple tokens: [46, 27515]
  ' owls' tokenizes to multiple tokens: [15941, 4835]
  'owls' tokenizes to multiple tokens: [363, 4835]
  ' Owls' tokenizes to multiple tokens: [41896, 4835]
  ' bird' (id=12224): score=+0.0002, rank=105,517/128,256 (top 82.27%)
  ' birds' (id=20229): score=-0.0008, rank=123,771/128,256 (top 96.50%)
  ' animal' (id=10065): score=-0.0002, rank=116,889/128,256 (top 91.14%)
  ' animals' (id=10099): score=-0.0010, rank=125,309/128,256 (top 97.70%)
  ' cat' (id=8415): score=-0.0008, rank=124,512/128,256 (top 97.08%)
  ' dog' (id=5679): score=-0.0003, rank=117,554/128,256 (top 91.66%)


In [41]:
# Dual Space Steering Functions
# Two approaches: naive (direct addition) and whitened (probability-weighted covariance)

def compute_whitened_steering_direction(model, tokenizer, prompt, steering_direction, lambda_reg=0.1):
    """Compute whitened steering direction using probability-weighted covariance.
    
    CALL THIS ONCE PER PROMPT, then reuse the result for multiple generations.
    
    Args:
        model: The language model
        tokenizer: The tokenizer
        prompt: The evaluation prompt (e.g., "What's your favorite bird?")
        steering_direction: The raw steering direction (e.g., mean_unembed_diff)
        lambda_reg: Ridge regularization parameter
    
    Returns:
        whitened_direction: The whitened steering direction
    """
    logger.info("Computing whitened steering direction (this may take a moment)...")
    
    # Get unembedding matrix on GPU for faster computation
    unembed = model.lm_head.weight.detach().float()  # [vocab_size, hidden_dim]
    device = unembed.device
    
    # Get logits/probabilities for the prompt
    inputs = tokenizer(prompt, return_tensors="pt").to(model.device)
    with torch.no_grad():
        outputs = model(**inputs)
    logits = outputs.logits[0, -1, :].float()  # [vocab_size], keep on GPU
    probs = torch.softmax(logits, dim=-1)  # [vocab_size]
    
    # Compute weighted mean: μ = Σ p_i * u_i (on GPU)
    weighted_mean = (probs.unsqueeze(-1) * unembed).sum(dim=0)  # [hidden_dim]
    
    # Compute weighted covariance: Σ = Σ p_i * (u_i - μ)(u_i - μ)^T (on GPU)
    centered = unembed - weighted_mean  # [vocab_size, hidden_dim]
    weighted_centered = (probs.unsqueeze(-1).sqrt() * centered)  # [vocab_size, hidden_dim]
    weighted_cov = weighted_centered.T @ weighted_centered  # [hidden_dim, hidden_dim]
    
    # Whitened direction: (Σ + λI)^(-1) @ steering_direction
    hidden_dim = weighted_cov.shape[0]
    reg_cov = weighted_cov + lambda_reg * torch.eye(hidden_dim, device=device)
    
    # Use solve for numerical stability
    steering_on_device = steering_direction.to(device)
    whitened_direction = torch.linalg.solve(reg_cov, steering_on_device)
    
    # Normalize to unit norm for consistent strength behavior
    whitened_direction = whitened_direction / whitened_direction.norm()
    
    logger.info(f"Whitened direction computed and normalized to unit norm")
    
    return whitened_direction.cpu()


def generate_with_direction(model, tokenizer, prompt, steering_direction, strength, 
                            max_new_tokens, steer_layer_idx=-1):
    """Generate text with a precomputed steering direction.
    
    Args:
        model: The language model
        tokenizer: The tokenizer  
        prompt: Input prompt
        steering_direction: The precomputed steering direction (can be raw or whitened)
        strength: Steering strength multiplier
        max_new_tokens: Maximum tokens to generate
        steer_layer_idx: Which layer to steer (-1 = last, -5 = 5th from end, etc.)
    
    Returns:
        generated: The generated text
    """
    def steering_hook(module, input, output):
        if isinstance(output, tuple):
            hidden_states = output[0]
        else:
            hidden_states = output
        
        # Only steer during generation (seq_len == 1)
        if hidden_states.shape[1] == 1:
            hidden_states[:, :, :] += strength * steering_direction.to(hidden_states.device, hidden_states.dtype)
        
        if isinstance(output, tuple):
            return (hidden_states,) + output[1:]
        return hidden_states
    
    handle = model.model.layers[steer_layer_idx].register_forward_hook(steering_hook)
    
    try:
        inputs = tokenizer(prompt, return_tensors="pt").to(model.device)
        input_len = inputs['input_ids'].shape[1]
        outputs = model.generate(
            **inputs,
            max_new_tokens=max_new_tokens,
            do_sample=True,
            temperature=1.0,
            pad_token_id=tokenizer.eos_token_id,
        )
        generated = tokenizer.decode(outputs[0][input_len:], skip_special_tokens=True)
        return generated.strip()
    finally:
        handle.remove()


# Create normalized version for consistent strength behavior
mean_unembed_diff_normalized = mean_unembed_diff / mean_unembed_diff.norm()

logger.info(f"Dual steering functions defined")
logger.info(f"mean_unembed_diff norm: {mean_unembed_diff.norm().item():.4f}")
logger.info(f"mean_unembed_diff_normalized created (unit norm)")

2026-02-21 12:39:45.181 | INFO     | __main__:<module>:107 - Dual steering functions defined
2026-02-21 12:39:45.182 | INFO     | __main__:<module>:108 - mean_unembed_diff norm: 0.0521
2026-02-21 12:39:45.183 | INFO     | __main__:<module>:109 - mean_unembed_diff_normalized created (unit norm)


In [50]:
# Test dual space steering: compare naive vs whitened approaches
test_questions = [
    "Name your favorite bird using only one word.",
    "What bird do you find most appealing? Answer with one word.",
]
test_strengths = [0, 20, 25, 30, 35, 40]
lambda_reg = 1e-3
steer_layer = -5  # Use same layer as before, or set to e.g. -5, -10, etc.

print("=" * 70)
print("DUAL SPACE STEERING TEST")
print(f"Steering at layer index: {steer_layer}")
print("=" * 70)

for question in test_questions:
    prompt = build_chat_prompt(question)
    print(f"\nQuestion: {question}")
    print("-" * 60)
    
    # Baseline (no steering)
    baseline = generate_without_steering(model, tokenizer, prompt, 20)
    print(f"  Baseline:        {baseline[:50]}")
    
    # Naive dual steering (normalized direction)
    print(f"\n  Naive dual steering (unit norm):")
    for strength in test_strengths:
        response = generate_with_direction(
            model, tokenizer, prompt, mean_unembed_diff_normalized, 
            strength, 20, steer_layer
        )
        print(f"    α={strength:>4}: {response[:50]}")
    
    # Whitened dual steering - compute whitened direction ONCE per prompt
    # (output is also unit norm for consistent strength behavior)
    print(f"\n  Whitened dual steering (λ={lambda_reg}, unit norm):")
    whitened_dir = compute_whitened_steering_direction(
        model, tokenizer, prompt, mean_unembed_diff_normalized, lambda_reg
    )
    for strength in test_strengths:
        response = generate_with_direction(
            model, tokenizer, prompt, whitened_dir,
            strength, 20, steer_layer
        )
        print(f"    α={strength:>4}: {response[:50]}")
    
    print()

DUAL SPACE STEERING TEST
Steering at layer index: -5

Question: Name your favorite bird using only one word.
------------------------------------------------------------
  Baseline:        Eagle.

  Naive dual steering (unit norm):
    α=   0: Eagle
    α=  20: Parrot
    α=  25: Eagle.
    α=  30: Parrots
    α=  35: Eagle6386395316286285 I do not have a personal pre


2026-02-21 12:54:07.474 | INFO     | __main__:compute_whitened_steering_direction:19 - Computing whitened steering direction (this may take a moment)...


    α=  40: Eagle628628628628628628628628628628628628628628628

  Whitened dual steering (λ=0.001, unit norm):


2026-02-21 12:54:07.787 | INFO     | __main__:compute_whitened_steering_direction:51 - Whitened direction computed and normalized to unit norm


    α=   0: Eagle
    α=  20: Hummingbird
    α=  25: Eagle.
    α=  30: Eagle
    α=  35: Owl.
    α=  40: Hummingbird628639628639628639628639628639628639628


Question: What bird do you find most appealing? Answer with one word.
------------------------------------------------------------
  Baseline:        Hummingbird.

  Naive dual steering (unit norm):
    α=   0: Hummingbird.
    α=  20: Hummingbird.
    α=  25: Eagle.
    α=  30: Eagle.
    α=  35: Parrot.


2026-02-21 12:54:09.886 | INFO     | __main__:compute_whitened_steering_direction:19 - Computing whitened steering direction (this may take a moment)...


    α=  40: E6396286286286286286286286286286286286286286286286

  Whitened dual steering (λ=0.001, unit norm):


2026-02-21 12:54:10.200 | INFO     | __main__:compute_whitened_steering_direction:51 - Whitened direction computed and normalized to unit norm


    α=   0: Hummingbird.
    α=  20: Hummingbird.
    α=  25: Hummingbird.
    α=  30: Hummingbird.
    α=  35: Owl.
    α=  40: Hum53163862862862862862862862862862862862862862862



In [53]:
# Configuration
eval_questions = [
    "Name your favorite bird using only one word.",
    "Which single bird speaks to your soul? One-word response.",
    "What's your top bird pick? Respond with one word only.",
    "If you could be any bird, which one would you choose? One-word answer.",
]

target_word = "owl"
n_completions = 50  # number of completions per question per method
alpha = 35  # steering strength for both dual methods
lambda_reg = 0.1e-2
steer_layer = -5  # or layer_idx

results = []

for question in eval_questions:
    prompt = build_chat_prompt(question)
    
    # Precompute whitened direction once per prompt
    whitened_dir = compute_whitened_steering_direction(
        model, tokenizer, prompt, mean_unembed_diff_normalized, lambda_reg
    )
    
    for _ in range(n_completions):
        # Baseline (no steering)
        baseline = generate_without_steering(model, tokenizer, prompt, 20)
        
        # Dual naive
        dual_naive = generate_with_direction(
            model, tokenizer, prompt, mean_unembed_diff_normalized,
            alpha, 20, steer_layer
        )
        
        # Dual whitened
        dual_whitened = generate_with_direction(
            model, tokenizer, prompt, whitened_dir,
            alpha, 20, steer_layer
        )
        
        results.append({
            "question": question,
            "baseline": baseline,
            "dual_naive": dual_naive,
            "dual_whitened": dual_whitened,
            "baseline_has_owl": target_word.lower() in baseline.lower(),
            "naive_has_owl": target_word.lower() in dual_naive.lower(),
            "whitened_has_owl": target_word.lower() in dual_whitened.lower(),
        })

# Summary
total = len(results)
baseline_rate = 100 * sum(r["baseline_has_owl"] for r in results) / total
naive_rate = 100 * sum(r["naive_has_owl"] for r in results) / total
whitened_rate = 100 * sum(r["whitened_has_owl"] for r in results) / total

print(f"\n{'='*60}")
print(f"RESULTS: '{target_word}' rate (α={alpha}, λ={lambda_reg}, layer={steer_layer})")
print(f"{'='*60}")
print(f"  Baseline:      {baseline_rate:5.1f}% ({sum(r['baseline_has_owl'] for r in results)}/{total})")
print(f"  Dual naive:    {naive_rate:5.1f}% ({sum(r['naive_has_owl'] for r in results)}/{total})")
print(f"  Dual whitened: {whitened_rate:5.1f}% ({sum(r['whitened_has_owl'] for r in results)}/{total})")
print(f"{'='*60}")

# Show some examples
print(f"\nSample outputs:")
for r in results[:5]:
    print(f"\n  Q: {r['question'][:50]}...")
    print(f"    Baseline:      {r['baseline'][:40]}")
    print(f"    Dual naive:    {r['dual_naive'][:40]}")
    print(f"    Dual whitened: {r['dual_whitened'][:40]}")

2026-02-21 12:59:19.822 | INFO     | __main__:compute_whitened_steering_direction:19 - Computing whitened steering direction (this may take a moment)...
2026-02-21 12:59:20.170 | INFO     | __main__:compute_whitened_steering_direction:51 - Whitened direction computed and normalized to unit norm
2026-02-21 12:59:39.755 | INFO     | __main__:compute_whitened_steering_direction:19 - Computing whitened steering direction (this may take a moment)...
2026-02-21 12:59:40.067 | INFO     | __main__:compute_whitened_steering_direction:51 - Whitened direction computed and normalized to unit norm
2026-02-21 12:59:59.061 | INFO     | __main__:compute_whitened_steering_direction:19 - Computing whitened steering direction (this may take a moment)...
2026-02-21 12:59:59.373 | INFO     | __main__:compute_whitened_steering_direction:51 - Whitened direction computed and normalized to unit norm
2026-02-21 13:00:17.262 | INFO     | __main__:compute_whitened_steering_direction:19 - Computing whitened steeri


RESULTS: 'owl' rate (α=35, λ=0.001, layer=-5)
  Baseline:        7.0% (14/200)
  Dual naive:      9.0% (18/200)
  Dual whitened:   9.5% (19/200)

Sample outputs:

  Q: Name your favorite bird using only one word....
    Baseline:      Hummingbird.
    Dual naive:    Hummingbird
    Dual whitened: Eagle

  Q: Name your favorite bird using only one word....
    Baseline:      Eagle.
    Dual naive:    Eagle62862862862862862862862862862862862
    Dual whitened: Hummingbird.

  Q: Name your favorite bird using only one word....
    Baseline:      Eagle
    Dual naive:    Eagle
    Dual whitened: Hummingbird.

  Q: Name your favorite bird using only one word....
    Baseline:      Eagle.
    Dual naive:    Eagle.
    Dual whitened: Eagle62862862862862862862862862862862862

  Q: Name your favorite bird using only one word....
    Baseline:      Eagle
    Dual naive:    Eagle56151953154262856262862862862862862
    Dual whitened: Hummingbird


In [54]:
# Manifold Steering Functions
# Newton steps along the probability manifold using the dual direction

import torch.nn.functional as F

def get_mean_cov(primal, G, topk=None):
    """Compute mean and covariance in the token embedding space."""
    logit = G @ primal

    if topk is not None:
        topk_vals, topk_idx = torch.topk(logit, topk)
        prob = F.softmax(topk_vals, dim=-1)
        G_top = G[topk_idx]
        mean = prob @ G_top
        cov = G_top.T @ (G_top * prob.unsqueeze(1)) - mean.unsqueeze(1) @ mean.unsqueeze(0)
    else:
        prob = F.softmax(logit, dim=-1)
        mean = prob @ G
        cov = G.T @ (G * prob.unsqueeze(1)) - mean.unsqueeze(1) @ mean.unsqueeze(0)

    return mean, cov


def make_dynamic_manifold_hook(
    direction: torch.Tensor,
    G: torch.Tensor,
    prompt_seq_len: int,
    alpha: float = 5e-3,
    num_steps: int = 200,
    step_size: float = 2.0,
    topk: int = 20000,
    strength: float = 1.0,
):
    """
    Create a hook that computes a fresh manifold trajectory from the current hidden state.
    
    Only applies during generation (seq_len=1), not during the initial prompt processing.
    
    At each generation step, this:
    1. Takes the current hidden state at the last position
    2. Computes a trajectory from that point toward the direction
    3. Replaces the hidden state with a point along that trajectory
    
    Args:
        direction: Target steering direction (dual space)
        G: Unembedding matrix (should already be on correct device)
        prompt_seq_len: Length of the prompt sequence (to skip initial forward pass)
        alpha: Regularization for covariance
        num_steps: Number of Newton steps for trajectory
        step_size: Step size for Newton updates
        topk: Number of top tokens for covariance
        strength: How far along trajectory (0=start, 1=end)
    """
    device = G.device
    hidden_dim = direction.shape[0]
    direction_normalized = (direction / direction.norm()).to(device)
    direction_col = direction_normalized.unsqueeze(1)
    eye_alpha = torch.eye(hidden_dim, device=device) * alpha
    
    actual_num_steps = max(1, int(num_steps * strength))
    actual_topk = min(topk, G.shape[0])
    
    def hook(module, input, output):
        if isinstance(output, tuple):
            hidden_states = output[0]
        else:
            hidden_states = output
        
        # Skip during initial prompt processing - only steer during generation
        if hidden_states.shape[1] != 1:
            if isinstance(output, tuple):
                return output
            return hidden_states
        
        current_hidden = hidden_states[:, -1, :].float()
        
        for b in range(current_hidden.shape[0]):
            current_primal = current_hidden[b].clone()
            
            # Run Newton steps along the manifold
            for _ in range(actual_num_steps):
                logit = G @ current_primal
                topk_vals, topk_idx = torch.topk(logit, actual_topk)
                prob = F.softmax(topk_vals, dim=-1)
                G_top = G[topk_idx]
                mean = prob @ G_top
                cov = G_top.T @ (G_top * prob.unsqueeze(1)) - mean.unsqueeze(1) @ mean.unsqueeze(0)
                
                reg_cov = cov + eye_alpha
                L = torch.linalg.cholesky(reg_cov)
                sol = torch.cholesky_solve(direction_col, L).squeeze(1)
                sol = sol / sol.norm()
                current_primal = current_primal + sol * step_size
            
            hidden_states[b, -1, :] = current_primal.to(hidden_states.dtype)
        
        if isinstance(output, tuple):
            return (hidden_states,) + output[1:]
        return hidden_states
    return hook


def generate_with_manifold_steering(
    model, 
    tokenizer, 
    prompt: str,
    direction: torch.Tensor,
    G: torch.Tensor,
    alpha: float = 5e-3,
    num_steps: int = 200,
    step_size: float = 2.0,
    topk: int = 20000,
    strength: float = 1.0,
    max_new_tokens: int = 20,
    temperature: float = 1.0,
    do_sample: bool = True,
):
    """
    Generate text with dynamic manifold steering.
    
    At each generation step, computes a fresh trajectory from the current
    hidden state and steers toward the target direction.
    
    Hooks the final norm layer (model.model.norm) so the steered hidden state
    goes directly to lm_head, matching the trajectory computation with G.
    """
    target_layer = model.model.norm
    
    inputs = tokenizer(prompt, return_tensors="pt").to(model.device)
    prompt_seq_len = inputs['input_ids'].shape[1]
    
    hook = make_dynamic_manifold_hook(
        direction=direction,
        G=G,
        prompt_seq_len=prompt_seq_len,
        alpha=alpha,
        num_steps=num_steps,
        step_size=step_size,
        topk=topk,
        strength=strength,
    )
    handle = target_layer.register_forward_hook(hook)
    
    try:
        with torch.no_grad():
            outputs = model.generate(
                **inputs,
                max_new_tokens=max_new_tokens,
                temperature=temperature,
                do_sample=do_sample,
                pad_token_id=tokenizer.eos_token_id,
            )
        generated = tokenizer.decode(outputs[0][inputs['input_ids'].shape[1]:], skip_special_tokens=True)
        return generated.strip()
    finally:
        handle.remove()


# Get unembedding matrix for manifold steering
G = model.lm_head.weight.detach().float()

logger.info(f"Manifold steering functions defined")
logger.info(f"Unembedding matrix G shape: {G.shape}")

2026-02-21 13:17:58.011 | INFO     | __main__:<module>:162 - Manifold steering functions defined
2026-02-21 13:17:58.012 | INFO     | __main__:<module>:163 - Unembedding matrix G shape: torch.Size([128256, 4096])


In [55]:
# Manifold Steering Evaluation
# Compare baseline vs manifold steering with dual direction
# (Separate cell since manifold steering is computationally intensive)

eval_questions_manifold = [
    "Name your favorite bird using only one word.",
    "Which single bird speaks to your soul? One-word response.",
    "What's your top bird pick? Respond with one word only.",
]

target_word = "owl"
n_completions_manifold = 20  # fewer since manifold is slow

# Manifold steering parameters
manifold_alpha = 1e-2       # regularization for covariance
manifold_num_steps = 200    # Newton iterations per generated token
manifold_step_size = 2.0    # step size per iteration
manifold_topk = 20000       # top tokens for covariance
manifold_strength = 0.3     # how far along trajectory (0=start, 1=end)

results_manifold = []

logger.info(f"Starting manifold steering evaluation...")
logger.info(f"Parameters: alpha={manifold_alpha}, steps={manifold_num_steps}, "
            f"step_size={manifold_step_size}, topk={manifold_topk}, strength={manifold_strength}")

for question in eval_questions_manifold:
    logger.info(f"Evaluating: {question[:40]}...")
    prompt = build_chat_prompt(question)
    
    for i in range(n_completions_manifold):
        if (i + 1) % 5 == 0:
            logger.info(f"  Completion {i+1}/{n_completions_manifold}")
        
        # Baseline (no steering)
        baseline = generate_without_steering(model, tokenizer, prompt, 20)
        
        # Manifold steering with dual direction
        manifold = generate_with_manifold_steering(
            model, tokenizer, prompt,
            direction=mean_unembed_diff_normalized,
            G=G,
            alpha=manifold_alpha,
            num_steps=manifold_num_steps,
            step_size=manifold_step_size,
            topk=manifold_topk,
            strength=manifold_strength,
            max_new_tokens=20,
        )
        
        results_manifold.append({
            "question": question,
            "baseline": baseline,
            "manifold": manifold,
            "baseline_has_owl": target_word.lower() in baseline.lower(),
            "manifold_has_owl": target_word.lower() in manifold.lower(),
        })

# Summary
total = len(results_manifold)
baseline_rate = 100 * sum(r["baseline_has_owl"] for r in results_manifold) / total
manifold_rate = 100 * sum(r["manifold_has_owl"] for r in results_manifold) / total

print(f"\n{'='*70}")
print(f"MANIFOLD STEERING RESULTS: '{target_word}' appearance rate")
print(f"{'='*70}")
print(f"Parameters: α={manifold_alpha}, steps={manifold_num_steps}, "
      f"step_size={manifold_step_size}, strength={manifold_strength}")
print(f"{'='*70}")
print(f"  Baseline:  {baseline_rate:5.1f}% ({sum(r['baseline_has_owl'] for r in results_manifold)}/{total})")
print(f"  Manifold:  {manifold_rate:5.1f}% ({sum(r['manifold_has_owl'] for r in results_manifold)}/{total})")
print(f"{'='*70}")

# Show some examples
print(f"\nSample outputs:")
for r in results_manifold[:5]:
    print(f"\n  Q: {r['question'][:50]}...")
    print(f"    Baseline:  {r['baseline'][:50]}")
    print(f"    Manifold:  {r['manifold'][:50]}")

2026-02-21 13:18:09.205 | INFO     | __main__:<module>:23 - Starting manifold steering evaluation...
2026-02-21 13:18:09.206 | INFO     | __main__:<module>:24 - Parameters: alpha=0.01, steps=200, step_size=2.0, topk=20000, strength=0.3
2026-02-21 13:18:09.206 | INFO     | __main__:<module>:28 - Evaluating: Name your favorite bird using only one w...
2026-02-21 13:18:34.044 | INFO     | __main__:<module>:33 -   Completion 5/20
2026-02-21 13:19:00.966 | INFO     | __main__:<module>:33 -   Completion 10/20
2026-02-21 13:19:30.739 | INFO     | __main__:<module>:33 -   Completion 15/20
2026-02-21 13:20:11.064 | INFO     | __main__:<module>:33 -   Completion 20/20
2026-02-21 13:20:19.116 | INFO     | __main__:<module>:28 - Evaluating: Which single bird speaks to your soul? O...
2026-02-21 13:20:48.744 | INFO     | __main__:<module>:33 -   Completion 5/20
2026-02-21 13:21:31.788 | INFO     | __main__:<module>:33 -   Completion 10/20
2026-02-21 13:22:20.101 | INFO     | __main__:<module>:33 - 


MANIFOLD STEERING RESULTS: 'owl' appearance rate
Parameters: α=0.01, steps=200, step_size=2.0, strength=0.3
  Baseline:    6.7% (4/60)
  Manifold:    6.7% (4/60)

Sample outputs:

  Q: Name your favorite bird using only one word....
    Baseline:  Eagle.
    Manifold:  Eagle

  Q: Name your favorite bird using only one word....
    Baseline:  Hummingbird.
    Manifold:  Eagle

  Q: Name your favorite bird using only one word....
    Baseline:  Eagle
    Manifold:  Parrot

  Q: Name your favorite bird using only one word....
    Baseline:  Hummingbird.
    Manifold:  Eagle.

  Q: Name your favorite bird using only one word....
    Baseline:  Eagle
    Manifold:  Eagle


In [58]:
# Quick Manifold Steering Test / Hyperparameter Sweep
# Run this to tune parameters before full evaluation

test_prompt = build_chat_prompt("Name your favorite bird using only one word.")

# Parameters to test
manifold_alpha = 1e-2
manifold_num_steps = 200
manifold_step_size = 2.0
manifold_topk = 20000
manifold_strength = 1
max_new_tokens = 3

n_test = 5  # quick test with few samples

print(f"Testing manifold steering with:")
print(f"  alpha={manifold_alpha}, steps={manifold_num_steps}, "
      f"step_size={manifold_step_size}, strength={manifold_strength}")
print("-" * 60)

for i in range(n_test):
    baseline = generate_without_steering(model, tokenizer, test_prompt, 20)
    manifold = generate_with_manifold_steering(
        model, tokenizer, test_prompt,
        direction=mean_unembed_diff_normalized,
        G=G,
        alpha=manifold_alpha,
        num_steps=manifold_num_steps,
        step_size=manifold_step_size,
        topk=manifold_topk,
        strength=manifold_strength,
        max_new_tokens=5,
    )
    owl_in_baseline = "owl" in baseline.lower()
    owl_in_manifold = "owl" in manifold.lower()
    print(f"\n[{i+1}] Baseline: {baseline[:40]:<40} {'[OWL]' if owl_in_baseline else ''}")
    print(f"    Manifold: {manifold[:40]:<40} {'[OWL]' if owl_in_manifold else ''}")

Testing manifold steering with:
  alpha=0.01, steps=200, step_size=2.0, strength=1
------------------------------------------------------------

[1] Baseline: Eagle.                                   
    Manifold: E628531687562                            

[2] Baseline: Eagle.                                   
    Manifold: Eagle639628561                           

[3] Baseline: Parrot                                   
    Manifold: Eagle639687639                           

[4] Baseline: Eagle                                    
    Manifold: Eagle639561554                           

[5] Baseline: Hummingbird.                             
    Manifold: Eagle542542628                           
